In [ ]:
!pip install open_clip_torch scikit-learn pillow -q
import torch, open_clip, numpy as np
from PIL import Image
from pathlib import Path
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
from google.colab import files
# Zip data/real_docs and data/genai_docs locally, upload here, unzip
!unzip -q docs_dataset.zip -d /content/data

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="openai")
model = model.to(device).eval()

def embed(path):
    img = preprocess(Image.open(path).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        return model.encode_image(img).cpu().numpy()[0]

X, y = [], []
for p in Path("/content/data/real_docs").glob("*"):
    X.append(embed(p)); y.append(0)
for p in Path("/content/data/genai_docs").glob("*"):
    X.append(embed(p)); y.append(1)
X = np.array(X); y = np.array(y)
print("features:", X.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
clf = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
auc = roc_auc_score(yte, clf.predict_proba(Xte)[:,1])
print("held-out AUC:", auc)
assert auc >= 0.75, "AUC below target — need more or more varied data"

In [ ]:
import torch.nn as nn
head = nn.Linear(512, 1)
head.weight.data = torch.tensor(clf.coef_, dtype=torch.float32)
head.bias.data   = torch.tensor(clf.intercept_, dtype=torch.float32)
torch.save({"state_dict": head.state_dict(), "auc": float(auc)},
           "clip_genai_probe.pt")
files.download("clip_genai_probe.pt")

In [ ]:
# Save the held-out raw scores + labels so calibration.py can fit Platt scaling
np.savez("genai_calibration_data.npz",
         scores=clf.predict_proba(Xte)[:,1], labels=yte)
files.download("genai_calibration_data.npz")